In [ ]:
import pandas as pd
df=pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
df.head()

In [ ]:
import pandas as pd
import numpy as np

def preprocess_spaceship(df, is_train=True):
    df = df.copy()
    
    # 1. Tách Cabin → Deck / Num / Side
    df[['Deck', 'CabinNum', 'Side']] = df['Cabin'].str.split('/', expand=True)
    df['CabinNum'] = pd.to_numeric(df['CabinNum'], errors='coerce')  # để NaN nếu lỗi
    df['Deck'] = df['Deck'].fillna('Unknown')
    df['Side'] = df['Side'].fillna('Unknown')
    
    # 2. Tách PassengerId → GroupId & GroupSize
    df['GroupId'] = df['PassengerId'].str.split('_').str[0]
    group_sizes = df['GroupId'].value_counts().to_dict()
    df['GroupSize'] = df['GroupId'].map(group_sizes)
    
    # 3. Tách Name → LastName & FamilySize / IsAlone
    df['LastName'] = df['Name'].str.split().str[-1]
    df['LastName'] = df['LastName'].fillna('Unknown')
    family_sizes = df['LastName'].value_counts().to_dict()
    df['FamilySize'] = df['LastName'].map(family_sizes)
    df['IsAlone'] = (df['FamilySize'] == 1) & (df['GroupSize'] == 1)
    
    # 4. Tổng chi tiêu
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df['TotalSpent'] = df[spend_cols].sum(axis=1)
    df['NoSpending'] = (df['TotalSpent'] == 0)
    
    # 5. Age bins
    bins = [0, 13, 25, 60, np.inf]
    labels = ['Child', 'Young', 'Adult', 'Senior']
    df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)
    df['AgeGroup'] = df['AgeGroup'].astype('object').fillna('Unknown')
    
    # 6. Categorical columns (dùng string để CatBoost tự xử lý tốt)
    cat_cols = [
        'HomePlanet', 'CryoSleep', 'Destination', 'VIP',
        'Deck', 'Side', 'AgeGroup', 'LastName',  # LastName có thể mạnh nếu giữ
        # 'GroupId' nếu muốn (nhưng thường drop vì cardinality cao)
    ]
    for col in cat_cols:
        df[col] = df[col].astype('object').fillna('Unknown')
    
    # 7. Numerical columns (để nguyên, CatBoost tự handle)
    num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
                'TotalSpent', 'GroupSize', 'FamilySize', 'CabinNum']
    
    # 8. Drop cột gốc không cần
    drop_cols = ['PassengerId', 'Cabin', 'Name']  # có thể giữ GroupId nếu muốn
    df = df.drop(columns=drop_cols, errors='ignore')
    
    # Nếu là test set → giữ nguyên cấu trúc
    if not is_train:
        # đảm bảo cùng cột với train
        pass
    
    return df, cat_cols + ['Deck', 'Side', 'AgeGroup']  # trả về df và list cat_features

# Sử dụng
train = df.copy()  # hoặc df của bạn
train_processed, cat_features = preprocess_spaceship(train, is_train=True)
test=pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')
test_processed,a=preprocess_spaceship(test, is_train=False)

print("Categorical features nên dùng:", cat_features)
print(train_processed.head())

In [ ]:
# cat_features = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

# print("Cat features tự động:", cat_features)

In [ ]:
from sklearn.model_selection import train_test_split

X = train_processed.drop(columns=['Transported'])   # adjust if target column has different name
y = train_processed['Transported'].astype(int)

X_train, X_test, y_train, y_test= train_test_split(X,y,train_size=0.8, random_state=42)
X_val, X_test, y_val, y_test= train_test_split(X_test, y_test, train_size=0.5, random_state=42)

In [ ]:
# import optuna
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.model_selection import StratifiedKFold, train_test_split
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
# from sklearn.calibration import CalibratedClassifierCV
# from sklearn.ensemble import StackingClassifier
# from sklearn.linear_model import LogisticRegression

# # Chỉ cần CatBoost
# from catboost import CatBoostClassifier

# # ────────────────────────────────────────────────
# # Hàm objective cho Optuna (giữ nguyên)
# # ────────────────────────────────────────────────
# def objective(trial):
#     params = {
#         'iterations': trial.suggest_int('iterations', 800, 5000),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
#         'depth': trial.suggest_int('depth', 4, 10),
#         'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
#         'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
#         'rsm': trial.suggest_float('rsm', 0.5, 1.0),
#         'random_strength': trial.suggest_float('random_strength', 0.1, 10.0, log=True),
#         'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide']),
#         'border_count': trial.suggest_int('border_count', 32, 255),
#         'eval_metric': 'Accuracy',
#         'random_state': 42,
#         'verbose': False,
#         'early_stopping_rounds': 200,
#         # 'task_type': 'GPU',  # bật nếu có GPU
#     }

#     bootstrap_type = trial.suggest_categorical('bootstrap_type', ['Bayesian'])
#     params['bootstrap_type'] = bootstrap_type

#     if bootstrap_type == 'Bayesian':
#         params['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0.0, 1.0)

#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#     scores = []

#     for train_idx, valid_idx in skf.split(X_train, y_train):
#         X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[valid_idx]
#         y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[valid_idx]

#         model = CatBoostClassifier(**params, cat_features=cat_features)

#         model.fit(
#             X_tr, y_tr,
#             eval_set=(X_va, y_va),
#             use_best_model=True,
#             verbose=False
#         )
#         pred = model.predict(X_va)
#         scores.append(accuracy_score(y_va, pred))

#     return np.mean(scores)


# # ────────────────────────────────────────────────
# # Chạy Optuna (giữ nguyên)
# # ────────────────────────────────────────────────
# study = optuna.create_study(
#     direction='maximize',
#     sampler=optuna.samplers.TPESampler(seed=42),
#     pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=20)
# )

# print("Đang tối ưu hyperparameter với Optuna + 5-fold CV...")
# study.optimize(objective, n_trials=40, timeout=3600*2)

# print("Best parameters:", study.best_params)
# print("Best CV Accuracy:", study.best_value)


# # ────────────────────────────────────────────────
# # Train Stacking CHỈ với CatBoost
# # ────────────────────────────────────────────────
# best_params = study.best_params

# # Base model: chỉ CatBoost
# cat = CatBoostClassifier(
#     **best_params,
#     cat_features=cat_features,
#     random_state=42,
#     verbose=100
# )

# estimators = [
#     ('catboost', cat),
#     # Bỏ XGBoost và LightGBM để tránh lỗi dtype với object columns
# ]

# stack = StackingClassifier(
#     estimators=estimators,
#     final_estimator=LogisticRegression(
#         max_iter=2000,           # tăng lên để tránh convergence warning
#         solver='lbfgs',
#         random_state=42,
#         n_jobs=-1
#     ),
#     cv=5,
#     stack_method='predict_proba',
#     passthrough=True,            # rất hữu ích khi chỉ có 1 base model
#     n_jobs=-1
# )

# print("Đang train Stacking Ensemble (chỉ CatBoost)...")
# stack.fit(X_train, y_train)


# # ────────────────────────────────────────────────
# # Calibration
# # ────────────────────────────────────────────────
# print("Đang calibrate probabilities...")
# calibrated_stack = CalibratedClassifierCV(estimator=stack, method='isotonic', cv=3)
# calibrated_stack.fit(X_val, y_val)


# # ────────────────────────────────────────────────
# # Đánh giá
# # ────────────────────────────────────────────────
# y_pred = calibrated_stack.predict(X_val)
# y_proba = calibrated_stack.predict_proba(X_val)

# print("\n" + "="*60)
# print("KẾT QUẢ CUỐI CÙNG (Stacking CatBoost + Calibrated)")
# print("="*60)
# print(f"Accuracy : {accuracy_score(y_val, y_pred):.5f}")
# print(f"Precision: {precision_score(y_val, y_pred, average='weighted'):.5f}")
# print(f"Recall   : {recall_score(y_val, y_pred, average='weighted'):.5f}")
# print(f"F1 Score : {f1_score(y_val, y_pred, average='weighted'):.5f}")
# print("\nClassification Report:")
# print(classification_report(y_val, y_pred))


# # Confusion Matrix
# plt.figure(figsize=(8,6))
# sns.heatmap(confusion_matrix(y_val, y_pred), annot=True, fmt='d', cmap='Blues')
# plt.title('Confusion Matrix - Stacking Calibrated (CatBoost only)')
# plt.show()


# # Feature Importance từ CatBoost
# cat_final = stack.named_estimators_['catboost']
# feat_imp = pd.DataFrame({
#     'feature': X_train.columns,
#     'importance': cat_final.feature_importances_
# }).sort_values('importance', ascending=False)

# plt.figure(figsize=(10, 8))
# sns.barplot(data=feat_imp.head(20), x='importance', y='feature')
# plt.title('Top 20 Feature Importance (CatBoost)')
# plt.show()

CODE TRAIN CHÍNH

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression   # ← Quan trọng: phải import

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ========================== BEST PARAMETERS ==========================
best_params = {
    'iterations': 3641,
    'learning_rate': 0.063972521148937,
    'depth': 7,
    'l2_leaf_reg': 3.378828264538197,
    'min_data_in_leaf': 22,
    'rsm': 0.8547508077448374,
    'random_strength': 4.894057592170787,
    'grow_policy': 'SymmetricTree',
    'border_count': 164,
    'bootstrap_type': 'Bayesian',
    'bagging_temperature': 0.1499228256998617
}

# Thêm các tham số training quan trọng (không có trong Optuna)
best_params.update({
    'eval_metric': 'Accuracy',
    'random_state': 42,
    'early_stopping_rounds': 200,
    'verbose': 100,
    # 'task_type': 'GPU',          # ← Uncomment nếu có GPU
    # 'devices': '0'
})

# ========================== Train CatBoost đơn lẻ (Khuyến nghị trước) ==========================
cat_model = CatBoostClassifier(
    **best_params,
    cat_features=cat_features
)

print("Đang train CatBoost với best parameters...")
cat_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True
)

# Đánh giá CatBoost đơn lẻ
y_pred_cat = cat_model.predict(X_val)
print("\n" + "="*50)
print("KẾT QUẢ CATBOOST ĐƠN LẺ")
print("="*50)
print(f"Accuracy : {accuracy_score(y_val, y_pred_cat):.5f}")
print(f"Precision: {precision_score(y_val, y_pred_cat, average='weighted'):.5f}")
print(f"Recall   : {recall_score(y_val, y_pred_cat, average='weighted'):.5f}")
print(f"F1 Score : {f1_score(y_val, y_pred_cat, average='weighted'):.5f}")

# Feature Importance
feat_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': cat_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=feat_imp.head(20), x='importance', y='feature')
plt.title('Top 20 Feature Importance (CatBoost)')
plt.tight_layout()
plt.show()

In [ ]:
test_pred = cat_model.predict(test_processed)

# Chuyển thành True/False theo format Kaggle
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': test_pred.astype(bool)   # True/False
})

# ────────────────────────────────────────────────
# 7. Lưu file submission
# ────────────────────────────────────────────────
submission.to_csv('submission.csv', index=False)
print("\nĐã lưu file: submission.csv")